# LeetCode 1689 — Partitioning Into Minimum Number Of Deci-Binary Numbers bằng Python

Link đề: https://leetcode.com/problems/partitioning-into-minimum-number-of-deci-binary-numbers/ · Độ khó: **Medium** · Chủ đề: String, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minPartitions(self, n: str) -> int:
        ...
```

Lộ trình: **DP "đổi tiền" trên các số deci-binary** cho số nhỏ (đáp án chuẩn) → **bóc từng lớp** (mỗi lần trừ đi một số deci-binary có chữ số 1 ở mọi vị trí khác 0) → nhận ra đáp án **chính là chữ số lớn nhất** → viết gọn `int(max(n))` và chứng minh vì sao cận dưới = cận trên.

## 1. Mục tiêu học

1. Hiểu định nghĩa số **deci-binary** (mọi chữ số là 0 hoặc 1, không có số 0 đứng đầu) và cách cộng nhiều số như vậy **theo từng cột chữ số**.
2. Viết được lời giải vét cạn kiểu **DP đổi tiền** (unbounded coin change) để có đáp án chuẩn cho số nhỏ.
3. Chứng minh đáp án bằng **cận dưới** (chữ số lớn nhất `d` cần ít nhất `d` số) và **cận trên** (xây dựng tường minh `d` số).
4. Nhận ra rằng khi cộng các số chỉ có chữ số 0/1 với **không quá 9 số**, phép cộng **không có nhớ** — lý do khiến bài toán tách được theo từng cột.
5. Xử lý input là **chuỗi** dài tới `10⁵` ký tự mà không đổi sang số nguyên lớn.

## 2. Đề bài & ví dụ

Một số thập phân gọi là **deci-binary** nếu mỗi chữ số của nó là `0` hoặc `1` và không có số `0` vô nghĩa ở đầu. Ví dụ `101`, `1100` là deci-binary; `112`, `3001` thì không.

Cho chuỗi `n` biểu diễn một số nguyên dương thập phân. Trả về **số lượng ít nhất** các số deci-binary **dương** có tổng bằng `n`.

**Ví dụ 1:** `n = "32"` → `3` vì `10 + 11 + 11 = 32`.

**Ví dụ 2:** `n = "82734"` → `8`.

**Ví dụ 3:** `n = "27346209830709182346"` → `9`.

**Ràng buộc:**
- `1 ≤ n.length ≤ 10⁵`
- `n` chỉ gồm chữ số, không có số 0 đứng đầu, biểu diễn số nguyên dương.

In [1]:
import random
import time
from functools import lru_cache
from typing import List

examples = [
    ("32", 3),
    ("82734", 8),
    ("27346209830709182346", 9),
]
for n, want in examples:
    print(f"n={n!r} -> kỳ vọng {want}")

n='32' -> kỳ vọng 3
n='82734' -> kỳ vọng 8
n='27346209830709182346' -> kỳ vọng 9


## 3. Kiến thức nền

### 3.1 Duyệt chuỗi chữ số và đổi ký tự sang số

Trong Python, `int("7") == 7`, hoặc nhanh hơn `ord(ch) - ord('0')`. So sánh hai **ký tự chữ số** bằng `<`, `>` cho cùng kết quả như so sánh số vì mã ASCII của `'0'..'9'` tăng dần.

In [2]:
s = "82734"
print([int(ch) for ch in s])
print([ord(ch) - ord('0') for ch in s])
print("'8' > '7' ?", '8' > '7', "| max ký tự =", max(s), "| int(max(s)) =", int(max(s)))

[8, 2, 7, 3, 4]
[8, 2, 7, 3, 4]
'8' > '7' ? True | max ký tự = 8 | int(max(s)) = 8


### 3.2 Kiểm tra một số có phải deci-binary

Chuyển số sang chuỗi và kiểm tra tập ký tự ⊆ `{'0', '1'}`. Số dương nên chữ số đầu luôn là `'1'` (Python không in số 0 đầu).

In [3]:
def is_deci_binary(x: int) -> bool:
    return x > 0 and set(str(x)) <= {"0", "1"}

print([(x, is_deci_binary(x)) for x in [101, 1100, 112, 3001, 1, 10, 0]])

[(101, True), (1100, True), (112, False), (3001, False), (1, True), (10, True), (0, False)]


### 3.3 Sinh mọi số deci-binary ≤ N

Số deci-binary có `k` chữ số tương ứng với chuỗi nhị phân độ dài `k` bắt đầu bằng 1. Nên ta có thể sinh chúng bằng cách duyệt `m = 1, 2, 3, …` và đọc `bin(m)[2:]` như một số **thập phân**.

In [4]:
def deci_binaries_upto(N):
    res, m = [], 1
    while True:
        x = int(bin(m)[2:])          # '101' (nhị phân của 5) đọc như thập phân = 101
        if x > N:
            break
        res.append(x)
        m += 1
    return res

print(deci_binaries_upto(120))

[1, 10, 11, 100, 101, 110, 111]


### 3.4 Phép cộng theo cột (không nhớ)

Khi cộng `k ≤ 9` số chỉ chứa chữ số 0/1, mỗi cột có tổng ≤ 9 nên **không bao giờ phát sinh nhớ**. Vì vậy chữ số ở cột `j` của tổng chính là **số lượng số có chữ số 1 ở cột `j`**.

In [5]:
nums = [10, 11, 11]
width = 2
for j in range(width):
    col = [str(x).zfill(width)[j] for x in nums]
    print(f"cột {j}: {col} -> tổng cột = {sum(map(int, col))}")
print("tổng =", sum(nums))

cột 0: ['1', '1', '1'] -> tổng cột = 3
cột 1: ['0', '1', '1'] -> tổng cột = 2
tổng = 32


### 3.4b Khi nào phép cộng **có** nhớ?

Nếu cộng từ 10 số trở lên có chữ số 1 ở cùng một cột, tổng cột ≥ 10 sinh nhớ sang cột bên trái. Ví dụ cộng mười lần số `1` được `10` — chữ số hàng chục là 1 dù **không** số nào có chữ số 1 ở hàng chục. Điều này cho thấy lập luận "chữ số cột = số lượng số có bit 1" chỉ đúng khi dùng **≤ 9 số**. May mắn là đáp án không bao giờ vượt 9, nên ta chỉ cần xét trường hợp đó.

In [6]:
nums = [1] * 10
print("10 lần số 1 =", sum(nums), "-> hàng chục có chữ số 1 nhờ NHỚ")
nums = [11] * 10
print("10 lần số 11 =", sum(nums), "-> cột đơn vị = 10 sinh nhớ, cột chục = 10 + 1 nhớ = 11 -> lại nhớ")
# nhưng dùng 10 số để tạo 10 là lãng phí: chính 10 là deci-binary (1 số)
print("is_deci_binary(10) =", is_deci_binary(10))

10 lần số 1 = 10 -> hàng chục có chữ số 1 nhờ NHỚ
10 lần số 11 = 110 -> cột đơn vị = 10 sinh nhớ, cột chục = 10 + 1 nhớ = 11 -> lại nhớ
is_deci_binary(10) = True


### 3.5 `lru_cache` cho DP đệ quy

Ta sẽ dùng `@lru_cache` để viết DP "đổi tiền": `f(x)` = số ít nhất số deci-binary cộng lại bằng `x`.

## 4. Ý tưởng nền tảng / trực giác

Tính tay `n = "32"`:
- Cột hàng chục có chữ số `3` ⇒ cần **ít nhất 3** số có chữ số 1 ở hàng chục (mỗi số deci-binary đóng góp tối đa 1 vào cột này, và không có nhớ từ cột hàng đơn vị vì mỗi cột ≤ 9).
- Cột hàng đơn vị có chữ số `2` ⇒ trong 3 số đó, chọn 2 số có chữ số 1 ở hàng đơn vị.
- Kết quả: `11 + 11 + 10 = 32`. Dùng đúng **3** số.

Ví dụ 2 `"82734"`: chữ số lớn nhất là `8`. Xây dựng 8 số, số thứ `k` (k = 1..8) có chữ số 1 ở cột `j` khi và chỉ khi `n[j] ≥ k`.

In [7]:
def build_parts(n: str):
    d = int(max(n))
    parts = []
    for k in range(1, d + 1):
        s = "".join("1" if int(ch) >= k else "0" for ch in n)
        parts.append(int(s))          # int() tự bỏ số 0 đứng đầu
    return parts

for n, _ in examples[:2]:
    p = build_parts(n)
    print(n, "=", " + ".join(map(str, p)), "| kiểm tra:", sum(p) == int(n), "| số lượng:", len(p))

32 = 11 + 11 + 10 | kiểm tra: True | số lượng: 3
82734 = 11111 + 11111 + 10111 + 10101 + 10100 + 10100 + 10100 + 10000 | kiểm tra: True | số lượng: 8


Bảng theo cột cho `"82734"` (hàng = số thứ `k`, cột = vị trí chữ số; ô = 1 nếu `n[j] ≥ k`):

| k | 8 | 2 | 7 | 3 | 4 | số |
|---|---|---|---|---|---|---|
| 1 | 1 | 1 | 1 | 1 | 1 | 11111 |
| 2 | 1 | 1 | 1 | 1 | 1 | 11111 |
| 3 | 1 | 0 | 1 | 1 | 1 | 10111 |
| 4 | 1 | 0 | 1 | 0 | 1 | 10101 |
| 5 | 1 | 0 | 1 | 0 | 0 | 10100 |
| 6 | 1 | 0 | 1 | 0 | 0 | 10100 |
| 7 | 1 | 0 | 1 | 0 | 0 | 10100 |
| 8 | 1 | 0 | 0 | 0 | 0 | 10000 |
| **tổng cột** | 8 | 2 | 7 | 3 | 4 | 82734 |

Hình dung như **biểu đồ cột**: mỗi chữ số là chiều cao một cột, ta "cắt ngang" từng tầng; số tầng bằng chiều cao cột cao nhất.

Mỗi số sinh ra có ít nhất một chữ số 1 (ở cột có chữ số lớn nhất) nên là **dương**; `int()` bỏ các số 0 đứng đầu nên không vi phạm "không có số 0 đầu".

## 5. Cách 1 — Brute Force / baseline: DP đổi tiền

**Ý tưởng:** coi mỗi số deci-binary `≤ N` là một "đồng xu" có thể dùng nhiều lần. Bài toán trở thành **coin change**: dùng ít đồng xu nhất để có tổng `N`.

`f(0) = 0`, `f(x) = 1 + min_{c ≤ x, c deci-binary} f(x − c)`.

Chỉ dùng được khi `N` nhỏ (vài nghìn) vì phải lập bảng cho mọi `x ≤ N`.

In [8]:
def min_partitions_dp(n: str) -> int:
    N = int(n)
    coins = deci_binaries_upto(N)
    INF = float("inf")
    f = [0] + [INF] * N
    for x in range(1, N + 1):
        best = INF
        for c in coins:
            if c > x:
                break
            if f[x - c] + 1 < best:
                best = f[x - c] + 1
        f[x] = best
    return f[N]

for n, want in examples[:1]:
    print(n, "->", min_partitions_dp(n), "kỳ vọng", want)
print("12345 ->", min_partitions_dp("12345"))

32 -> 3 kỳ vọng 3
12345 -> 5


**Độ phức tạp:** với `N` là giá trị số, số đồng xu khoảng `2^{số chữ số} ≈ N^{log10 2}`; tổng thời gian `O(N · N^{0.3})`. Không thể dùng cho `"82734"` một cách thoải mái, càng không với chuỗi dài `10⁵` chữ số (giá trị cỡ `10^{100000}`).

**Nút thắt:** DP làm việc theo **giá trị** của số. Cần chuyển sang suy nghĩ theo **chữ số**.

In [9]:
for n in ["999", "4567", "20000"]:
    t0 = time.perf_counter()
    r = min_partitions_dp(n)
    print(f"n={n}: {r}  ({time.perf_counter() - t0:.3f}s)")

n=999: 9  (0.000s)
n=4567: 7  (0.003s)
n=20000: 2  (0.017s)


## 6. Ý tưởng thuật toán / bước đột phá

**Cận dưới.** Gọi `d` là chữ số lớn nhất của `n`, nằm ở cột `j`. Giả sử dùng `k` số deci-binary. Ta chứng minh `k ≥ d`:
- Nếu `k ≥ 10` thì hiển nhiên `k ≥ 9 ≥ d`.
- Nếu `k ≤ 9`, mọi cột có tổng ≤ 9 nên **không có nhớ**; chữ số cột `j` của tổng = số lượng số có bit 1 ở cột `j` ≤ `k`. Vậy `d ≤ k`.

**Cận trên.** Mục 4 đã xây dựng tường minh `d` số: số thứ `k` có chữ số 1 ở cột `j` ⇔ `n[j] ≥ k`. Cột `j` nhận đúng `n[j]` chữ số 1 ⇒ tổng đúng bằng `n`.

**Kết luận:** đáp án = **chữ số lớn nhất** của `n`.

**Quy tắc nếu… thì…:**
- Nếu gặp chữ số `'9'` thì có thể dừng ngay, đáp án là 9 (không có chữ số nào lớn hơn).
- Nếu chuỗi chỉ gồm `0/1` thì đáp án là 1 (chính `n` là deci-binary).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Bóc từng lớp (mô phỏng tham lam)

Mỗi vòng, trừ đi số deci-binary có chữ số 1 ở **mọi** cột còn khác 0. Đếm số vòng tới khi mọi chữ số bằng 0. Đây là cách "làm tay" đúng như mục 4, chạy theo mảng chữ số nên không cần số lớn.

In [10]:
def min_partitions_peel(n: str) -> int:
    digits = [int(ch) for ch in n]
    rounds = 0
    while any(digits):
        for j in range(len(digits)):
            if digits[j] > 0:
                digits[j] -= 1
        rounds += 1
    return rounds

for n, want in examples:
    print(n, "->", min_partitions_peel(n), "kỳ vọng", want)

32 -> 3 kỳ vọng 3
82734 -> 8 kỳ vọng 8
27346209830709182346 -> 9 kỳ vọng 9


**Độ phức tạp:** tối đa 9 vòng, mỗi vòng `O(L)` (L = độ dài chuỗi) ⇒ Time `O(9L) = O(L)`, Space `O(L)` cho mảng chữ số. Đã đủ nhanh, nhưng còn có thể gọn hơn nhiều.

### 7.2 Cách 3 — Lấy chữ số lớn nhất bằng vòng lặp (có dừng sớm)

Duyệt từng ký tự, giữ `best`. Gặp `'9'` thì trả về 9 ngay.

In [11]:
def min_partitions_loop(n: str) -> int:
    best = 0
    for ch in n:
        v = ord(ch) - 48
        if v > best:
            best = v
            if best == 9:
                break
    return best

for n, want in examples:
    print(n, "->", min_partitions_loop(n), "kỳ vọng", want)

32 -> 3 kỳ vọng 3
82734 -> 8 kỳ vọng 8
27346209830709182346 -> 9 kỳ vọng 9


Code nộp LeetCode của Cách 3:

```python
class Solution:
    def minPartitions(self, n: str) -> int:
        best = 0
        for ch in n:
            v = ord(ch) - 48
            if v > best:
                best = v
                if best == 9:
                    break
        return best
```

**Độ phức tạp:** Time `O(L)` worst, Space `O(1)`.

### 7.3 Cách 4 — Một dòng: `int(max(n))`

Vì so sánh ký tự chữ số tương đương so sánh số, `max(n)` trả về **ký tự** chữ số lớn nhất. Hàm `max` cài bằng C nên rất nhanh.

In [12]:
def min_partitions_max(n: str) -> int:
    return int(max(n))

for n, want in examples:
    print(n, "->", min_partitions_max(n), "kỳ vọng", want)

32 -> 3 kỳ vọng 3
82734 -> 8 kỳ vọng 8
27346209830709182346 -> 9 kỳ vọng 9


### 7.4 Cách 5 — Kiểm tra thành viên từ 9 xuống 1

`'9' in n` là phép tìm kiếm chuỗi cài bằng C. Thử lần lượt `'9'`, `'8'`, …; ký tự đầu tiên có mặt chính là đáp án. Worst case 9 lần quét, nhưng mỗi lần quét rất nhanh.

In [13]:
def min_partitions_in(n: str) -> int:
    for d in "987654321":
        if d in n:
            return int(d)
    return 0          # không xảy ra vì n là số dương

for n, want in examples:
    print(n, "->", min_partitions_in(n), "kỳ vọng", want)

32 -> 3 kỳ vọng 3
82734 -> 8 kỳ vọng 8
27346209830709182346 -> 9 kỳ vọng 9


**Độ phức tạp:** Time `O(9L) = O(L)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 2 (bóc lớp) trên `n = "32"` và `n = "82734"`:

| vòng | chữ số trước | trừ 1 ở cột khác 0 | chữ số sau |
|---|---|---|---|
| 1 | `3 2` | `1 1` | `2 1` |
| 2 | `2 1` | `1 1` | `1 0` |
| 3 | `1 0` | `1 0` | `0 0` |

Ba vòng ⇒ đáp án 3. Các số đã trừ: `11, 11, 10`.

In [14]:
def trace_peel(n: str):
    digits = [int(ch) for ch in n]
    rounds = 0
    print(f"bắt đầu: {digits}")
    while any(digits):
        part = "".join("1" if x > 0 else "0" for x in digits)
        digits = [x - 1 if x > 0 else 0 for x in digits]
        rounds += 1
        print(f"vòng {rounds}: trừ {int(part):>6}  -> còn {digits}")
    print("số vòng =", rounds)
    return rounds

trace_peel("32")

bắt đầu: [3, 2]
vòng 1: trừ     11  -> còn [2, 1]
vòng 2: trừ     11  -> còn [1, 0]
vòng 3: trừ     10  -> còn [0, 0]
số vòng = 3


3

In [15]:
trace_peel("82734")

bắt đầu: [8, 2, 7, 3, 4]
vòng 1: trừ  11111  -> còn [7, 1, 6, 2, 3]
vòng 2: trừ  11111  -> còn [6, 0, 5, 1, 2]
vòng 3: trừ  10111  -> còn [5, 0, 4, 0, 1]
vòng 4: trừ  10101  -> còn [4, 0, 3, 0, 0]
vòng 5: trừ  10100  -> còn [3, 0, 2, 0, 0]
vòng 6: trừ  10100  -> còn [2, 0, 1, 0, 0]
vòng 7: trừ  10100  -> còn [1, 0, 0, 0, 0]
vòng 8: trừ  10000  -> còn [0, 0, 0, 0, 0]
số vòng = 8


8

In [16]:
def trace_loop(n: str):
    best = 0
    for i, ch in enumerate(n):
        v = ord(ch) - 48
        upd = v > best
        if upd:
            best = v
        print(f"i={i:2d} ch={ch} best={best}{'  <- cập nhật' if upd else ''}")
        if best == 9:
            print("gặp 9 -> dừng sớm"); break
    return best

trace_loop("27346209830709182346")

i= 0 ch=2 best=2  <- cập nhật
i= 1 ch=7 best=7  <- cập nhật
i= 2 ch=3 best=7
i= 3 ch=4 best=7
i= 4 ch=6 best=7
i= 5 ch=2 best=7
i= 6 ch=0 best=7
i= 7 ch=9 best=9  <- cập nhật
gặp 9 -> dừng sớm


9

## 9. Code nộp LeetCode

In [17]:
class Solution:
    def minPartitions(self, n: str) -> int:
        return int(max(n))

sol = Solution()
for n, want in examples:
    got = sol.minPartitions(n)
    print(n, "->", got, "OK" if got == want else "SAI")

32 -> 3 OK
82734 -> 8 OK
27346209830709182346 -> 9 OK


## 10. Vì sao thuật toán đúng

Gọi `d = max chữ số của n`.

**(i) Không thể dùng ít hơn `d` số.** Giả sử có `k < d ≤ 9` số deci-binary có tổng `n`. Vì `k ≤ 8`, ở mọi cột tổng các chữ số ≤ 8 < 10 nên phép cộng không có nhớ; khi đó chữ số ở cột chứa `d` của tổng ≤ `k < d` — mâu thuẫn.

**(ii) Luôn dùng được đúng `d` số.** Với `k = 1..d`, đặt `p_k` là số có chữ số 1 ở cột `j` nếu `n[j] ≥ k`, ngược lại 0. Mỗi `p_k` có chữ số 1 ở cột chứa `d` nên dương; khi viết dưới dạng số nguyên, các số 0 đầu bị bỏ. Cột `j` có đúng `n[j]` số `p_k` mang chữ số 1, tổng cột = `n[j] ≤ 9` ⇒ không nhớ ⇒ `Σ p_k = n`.

Từ (i) và (ii), đáp án = `d`. Cách 2 tạo đúng các `p_k` theo thứ tự `k = 1, 2, …` (mỗi vòng trừ 1 ở các cột còn dương), nên số vòng cũng bằng `d`.

Lời giải tối ưu **không duy nhất**. Liệt kê mọi bộ (không tính thứ tự) gồm đúng `d` số deci-binary có tổng `N` cho vài `N` nhỏ: ngoài cách "cắt theo tầng" còn có những cách khác — nhưng không cách nào dùng ít hơn `d` số.

In [18]:
from itertools import combinations_with_replacement

def all_optimal(N):
    d = int(max(str(N)))
    coins = deci_binaries_upto(N)
    sols = [c for c in combinations_with_replacement(coins, d) if sum(c) == N]
    fewer = any(sum(c) == N for k in range(1, d) for c in combinations_with_replacement(coins, k))
    return d, sols, fewer

for N in [121, 222, 32]:
    d, sols, fewer = all_optimal(N)
    print(f"N={N}, d={d}: {len(sols)} cách tối ưu -> {sols} | có cách ít hơn d? {fewer}")

N=121, d=2: 2 cách tối ưu -> [(10, 111), (11, 110)] | có cách ít hơn d? False
N=222, d=2: 1 cách tối ưu -> [(111, 111)] | có cách ít hơn d? False
N=32, d=3: 1 cách tối ưu -> [(10, 11, 11)] | có cách ít hơn d? False


## 11. Phân tích Time / Space

Gọi `L = len(n) ≤ 10⁵`, `N` = giá trị số của `n`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. DP đổi tiền | `O(N · N^{0.3})` | `O(N)` | chỉ dùng được với `N` cỡ vài vạn |
| 2. Bóc lớp | `O(9L)` | `O(L)` | mô phỏng, dễ hình dung |
| 3. Vòng lặp + dừng sớm | `O(L)` worst, có thể sớm hơn | `O(1)` | không cấp phát |
| 4. `int(max(n))` | `O(L)` | `O(1)` | nhanh nhất thực tế (C) |
| 5. `'9' in n`… | `O(9L)` worst | `O(1)` | cũng chạy bằng C |

In [19]:
random.seed(1)
big = "1" + "".join(random.choice("01234567") for _ in range(10**5 - 1))   # không có 8, 9 -> không dừng sớm
for name, fn in [("peel", min_partitions_peel), ("loop", min_partitions_loop),
                 ("max", min_partitions_max), ("in", min_partitions_in)]:
    t0 = time.perf_counter(); r = fn(big)
    print(f"{name:4s}: {r}  {time.perf_counter() - t0:.4f}s")

peel: 7  0.0253s
loop: 7  0.0027s
max : 7  0.0012s
in  : 7  0.0000s


## 12. Bộ test

Đối chiếu mọi cách với DP vét cạn trên mọi `N ≤ 3000` (đầy đủ, không cần ngẫu nhiên), thêm edge case và kiểm tra cấu trúc lời giải (các số xây dựng đều là deci-binary và cộng lại đúng `n`) trên chuỗi dài ngẫu nhiên seed cố định.

In [20]:
fns = [min_partitions_peel, min_partitions_loop, min_partitions_max, min_partitions_in,
       lambda s: Solution().minPartitions(s)]

for n, want in examples:
    for fn in fns:
        assert fn(n) == want

# DP một lần cho mọi giá trị ≤ 3000
LIMIT = 3000
coins = deci_binaries_upto(LIMIT)
f = [0] + [10**9] * LIMIT
for x in range(1, LIMIT + 1):
    f[x] = 1 + min(f[x - c] for c in coins if c <= x)
for x in range(1, LIMIT + 1):
    for fn in fns:
        assert fn(str(x)) == f[x], x
print(f"mọi N trong [1, {LIMIT}] khớp DP vét cạn")

mọi N trong [1, 3000] khớp DP vét cạn


In [21]:
edge = [("1", 1), ("9", 9), ("10", 1), ("1000000", 1), ("11111", 1), ("99999", 9),
        ("1" + "0" * 99999, 1), ("2" * 100000, 2)]
for n, want in edge:
    for fn in fns:
        assert fn(n) == want, (n[:10], fn)

random.seed(42)
for _ in range(200):
    L = random.randint(1, 200)
    n = str(random.randint(1, 9)) + "".join(random.choice("0123456789") for _ in range(L - 1))
    parts = build_parts(n)
    assert all(is_deci_binary(p) for p in parts)
    assert sum(parts) == int(n)
    assert len(parts) == min_partitions_max(n)
print("edge case + kiểm tra xây dựng tường minh: OK")

edge case + kiểm tra xây dựng tường minh: OK

## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đếm số chữ số khác 0, hoặc tính tổng chữ số

Nhầm "mỗi chữ số cần một số riêng" ⇒ trả về tổng chữ số (quá lớn) hoặc số chữ số khác 0.

In [22]:
def wrong_digit_sum(n: str) -> int:
    return sum(int(ch) for ch in n)

for n, want in examples[:2]:
    print(n, "sai =", wrong_digit_sum(n), "| đúng =", want)

32 sai = 5 | đúng = 3
82734 sai = 24 | đúng = 8


Các số deci-binary **dùng chung** cho nhiều cột cùng lúc: `11` đóng góp vào cả hai cột. **Sửa:** chỉ cột "nặng" nhất quyết định đáp án ⇒ `max`.

### Lỗi 2 — `max(n)` mà quên đổi sang `int`

`max("32")` trả về **chuỗi** `'3'`. LeetCode yêu cầu `int`; so sánh `'3' == 3` là `False`.

In [23]:
def wrong_no_int(n: str):
    return max(n)

got = wrong_no_int("32")
print(repr(got), "| type:", type(got).__name__, "| bằng 3?", got == 3)

'3' | type: str | bằng 3? False


**Sửa:** `return int(max(n))`.

### Lỗi 3 — Đổi cả chuỗi sang `int` rồi làm toán

Chuỗi dài tới `10⁵` chữ số. Từ Python 3.11, `int()` trên chuỗi quá 4300 chữ số bị chặn (`ValueError`) để chống tấn công; kể cả khi được phép, chia/lấy dư lặp lại trên số khổng lồ tốn `O(L²)`.

In [24]:
import sys

def wrong_big_int(n: str) -> int:
    x = int(n)
    best = 0
    while x:
        best = max(best, x % 10)
        x //= 10
    return best

big = "7" * 5000
try:
    r = wrong_big_int(big)
    print("chạy được (giới hạn int đã tắt):", r)
except ValueError as e:
    print("ValueError:", str(e)[:90], "...")
print("cách đúng:", min_partitions_max(big))

ValueError: Exceeds the limit (4300 digits) for integer string conversion: value has 5000 digits; use  ...
cách đúng: 7


**Sửa:** làm việc trực tiếp trên ký tự của chuỗi.

### Lỗi 4 — Bóc lớp nhưng trừ 1 ở **mọi** cột (kể cả cột đã bằng 0)

Chữ số âm làm vòng lặp `while any(digits)` không dừng (số âm là truthy) hoặc trả sai. Ta giới hạn số vòng để minh họa mà không treo.

In [25]:
def wrong_peel(n: str, max_rounds=20):
    digits = [int(ch) for ch in n]
    rounds = 0
    while any(digits) and rounds < max_rounds:
        digits = [x - 1 for x in digits]     # SAI: cột 0 thành -1
        rounds += 1
    return rounds, digits

print("wrong_peel('32') =", wrong_peel("32"), "| đúng =", min_partitions_peel("32"))

wrong_peel('32') = (20, [-17, -18]) | đúng = 3


**Sửa:** chỉ trừ ở cột `> 0` (Cách 2).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "tổng của các số chỉ gồm chữ số 0/1" ⇒ nghĩ **theo cột**, phép cộng không nhớ khi số lượng ≤ 9.
- Bài Medium mà đáp án một dòng thường đòi **chứng minh cận dưới = cận trên** — hãy luyện thói quen này.
- Input là chuỗi số rất dài ⇒ **không** đổi sang `int`; xử lý ký tự.
- So sánh ký tự `'0'..'9'` giống so sánh số ⇒ `max`, `min`, `sorted` trên chuỗi chữ số dùng được ngay.
- Bài cùng họ "đáp án = max của một đại lượng theo cột/vị trí": **1689** (bài này), *Minimum Number of Increments on Subarrays to Form a Target Array* (1526, đáp án theo các bước tăng), *Bulb Switcher*… — đều có dạng "cắt theo tầng".
- Hình dung chữ số như **cột trong biểu đồ** và "cắt ngang từng tầng" là mẹo trực quan mạnh cho mọi bài "mỗi thao tác giảm 1 trên một tập vị trí".
- Khi nghi ngờ công thức, viết DP vét cạn cho `N ≤ vài nghìn` và so sánh **toàn bộ** (không chỉ ngẫu nhiên).

## 15. Biến thể của bài

1. **Trả về chính các số deci-binary** (không chỉ số lượng): hàm `build_parts` ở mục 4.
2. **Cơ số `b` bất kỳ**: số "deci-binary" trong cơ số `b` (chữ số 0/1) ⇒ đáp án vẫn là chữ số lớn nhất trong biểu diễn cơ số `b`.
3. **Chỉ được dùng các số deci-binary có đúng `L` chữ số** (bắt đầu bằng 1): cột đầu tiên phải có chữ số = số lượng số ⇒ đáp án tồn tại khi chữ số đầu ≥ mọi chữ số khác (đối chiếu lại đề gốc — đây là biến thể tự đặt).
4. **Tổng của các số "repunit" (11…1)**: bài khác hẳn, cần DP/tham lam theo độ dài.
5. **Đếm số cách** phân tích thành đúng `d` số deci-binary (không tính thứ tự): bài tổ hợp nâng cao.

In [26]:
def to_base(x, b):
    out = []
    while x:
        out.append(x % b); x //= b
    return out[::-1] or [0]

def min_parts_base(x, b):
    return max(to_base(x, b))

# kiểm tra biến thể 2 bằng DP trong cơ số 3 cho x nhỏ
def dp_base(LIMIT, b):
    coins = []
    m = 1
    while True:
        bits = bin(m)[2:]
        v = int(bits, b)                 # đọc chuỗi 0/1 như số cơ số b
        if v > LIMIT: break
        coins.append(v); m += 1
    f = [0] + [10**9] * LIMIT
    for x in range(1, LIMIT + 1):
        f[x] = 1 + min(f[x - c] for c in coins if c <= x)
    return f

f3 = dp_base(500, 3)
assert all(f3[x] == min_parts_base(x, 3) for x in range(1, 501))
print("biến thể cơ số 3 khớp DP cho x ≤ 500; ví dụ 100 =", to_base(100, 3), "->", min_parts_base(100, 3))

biến thể cơ số 3 khớp DP cho x ≤ 500; ví dụ 100 = [1, 0, 2, 0, 1] -> 2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DP đổi tiền | coin change trên số deci-binary | `O(N^{1.3})` | `O(N)` | kiểm chứng với số nhỏ |
| 2. Bóc lớp | trừ 1 ở cột dương mỗi vòng | `O(9L)` | `O(L)` | giải thích, in ra các số |
| 3. Vòng lặp + break | tìm max, dừng ở 9 | `O(L)` | `O(1)` | phỏng vấn (thể hiện tối ưu nhỏ) |
| 4. `int(max(n))` | max ký tự | `O(L)` | `O(1)` | **nộp bài** |
| 5. `d in n` từ 9 | tìm kiếm chuỗi C | `O(9L)` | `O(1)` | tương đương 4 |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `minPartitions("1010101")` và `minPartitions("90")` bằng bao nhiêu?

**Bài 2 (tính tay):** liệt kê các số deci-binary mà Cách 2 trừ đi với `n = "425"`.

**Bài 3 (điền TODO):** hoàn thiện `parts_of(n)` trả về **danh sách chuỗi** các số deci-binary (không có số 0 đầu) có tổng bằng `n` với số lượng ít nhất.

**Bài 4 (tự viết lại):** viết `min_partitions_counter(n)` dùng `collections.Counter` hoặc `set(n)` (không dùng `max` trực tiếp trên `n`).

**Bài 5 (mini challenge):** biến thể "**số trinary**" — các số chỉ có chữ số `0, 1, 2` (không số 0 đầu). Số ít nhất cần để có tổng `n` là bao nhiêu? Viết công thức, kiểm chứng bằng DP với mọi `N ≤ 2000`.

In [27]:
def parts_of(n: str) -> List[str]:
    # TODO: với k = 1..max digit, tạo chuỗi '1' nếu n[j] >= k ngược lại '0', bỏ số 0 đầu
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Chữ số lớn nhất của `"1010101"`? (2) Chuỗi chỉ có 0/1 thì bản thân nó là deci-binary. (3) Đáp án `1` và `9`.

**Bài 2:** (1) `d = 5` nên có 5 số. (2) Vòng 1: mọi cột dương ⇒ `111`. (3) Các số: `111, 111, 101, 101, 001 → 1`; tổng `425`.

**Bài 3:** (1) Tính `d = int(max(n))`. (2) Dùng `"".join(...)`. (3) `.lstrip("0")` để bỏ số 0 đầu.

**Bài 4:** (1) `set(n)` cho các chữ số xuất hiện. (2) Duyệt `'9'` → `'1'`. (3) Trả về chữ số đầu tiên có trong tập.

**Bài 5:** (1) Mỗi số đóng góp tối đa 2 vào một cột. (2) Với `k` số và `2k ≤ 9`, không có nhớ ⇒ cần `2k ≥ d`. (3) Đáp án `⌈d / 2⌉`; kiểm tra lại cận trên bằng cách xây dựng.

## 19. Đáp án tham khảo

In [28]:
from collections import Counter

# Bài 1
assert Solution().minPartitions("1010101") == 1 and Solution().minPartitions("90") == 9

# Bài 2
assert build_parts("425") == [111, 111, 101, 101, 1] and sum(build_parts("425")) == 425

# Bài 3
def parts_of_ans(n: str) -> List[str]:
    d = int(max(n))
    return ["".join("1" if ch >= str(k) else "0" for ch in n).lstrip("0") for k in range(1, d + 1)]

for n, want in examples:
    p = parts_of_ans(n)
    assert len(p) == want and sum(map(int, p)) == int(n) and all(set(x) <= {"0", "1"} and x[0] == "1" for x in p)

# Bài 4
def min_partitions_counter(n: str) -> int:
    present = set(n)
    for d in "987654321":
        if d in present:
            return int(d)
    return 0
for x in range(1, 3001):
    assert min_partitions_counter(str(x)) == f[x]
print("Bài 1-4: OK")

Bài 1-4: OK


In [29]:
# Bài 5 — số trinary (chữ số 0/1/2)
def min_trinary(n: str) -> int:
    d = int(max(n))
    return (d + 1) // 2

LIM = 2000
coins3 = []
m = 1
while True:
    s = ""
    t = m
    while t:
        s = str(t % 3) + s; t //= 3
    v = int(s)                    # chuỗi chữ số 0/1/2 đọc như thập phân
    if v > LIM: break
    coins3.append(v); m += 1
g = [0] + [10**9] * LIM
for x in range(1, LIM + 1):
    g[x] = 1 + min(g[x - c] for c in coins3 if c <= x)
assert all(g[x] == min_trinary(str(x)) for x in range(1, LIM + 1))
print("Bài 5: công thức ceil(d/2) khớp DP với mọi N ≤", LIM, "| ví dụ '97' ->", min_trinary("97"))

Bài 5: công thức ceil(d/2) khớp DP với mọi N ≤ 2000 | ví dụ '97' -> 5


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def minPartitions(self, n: str) -> int:
        return int(max(n))      # đáp án = chữ số lớn nhất
```

Khung tư duy: **cận dưới** (cột có chữ số `d` cần ≥ `d` số, vì không có nhớ khi ≤ 9 số) + **cận trên** (xây dựng `d` số theo ngưỡng `n[j] ≥ k`).

**Checklist trước khi submit:**
- [ ] Trả về `int`, không phải ký tự.
- [ ] Không đổi cả chuỗi `n` sang `int` (chuỗi dài tới `10⁵`).
- [ ] Hiểu vì sao không phải tổng chữ số / số chữ số khác 0.
- [ ] Đã thử `"1"`, `"10"`, `"9"`, chuỗi toàn `0/1`.
- [ ] (Tùy chọn) dừng sớm khi gặp `'9'`.